# 05 — Experimentos com instâncias Large

Objetivo: executar o PBIL-Fuzzy nas instâncias Large, registrar Cmax, tempo, evolução de alpha/beta e diversidade estrutural, e gerar uma tabela consolidada para análise posterior.

In [ ]:
import os
import sys
import time
import copy
import json

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

caminho_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
if caminho_raiz not in sys.path:
    sys.path.insert(0, caminho_raiz)

In [ ]:
from src.core.instance import carregar_instancia, listar_instancias
from src.engine.pbil_fuzzy import PBILFuzzy
from src.utils import visualization

In [ ]:
def carregar_configuracao(caminho_config):
    with open(caminho_config, "r", encoding="utf-8") as arquivo:
        return yaml.safe_load(arquivo)


def preparar_config_large(config_original, max_geracoes=None, n_pop=None):
    config_execucao = copy.deepcopy(config_original)

    if max_geracoes is not None:
        config_execucao["criterio_parada"]["max_geracoes"] = max_geracoes

    if n_pop is not None:
        config_execucao["pbil"]["n_pop"] = n_pop

    return config_execucao


def calcular_semente_execucao(semente_base, indice_instancia, indice_execucao):
    return int(semente_base + 1000 * indice_instancia + indice_execucao)


def executar_rodada_large(caminho_instancia, config_execucao, indice_instancia, indice_execucao):
    instancia = carregar_instancia(caminho_instancia)

    semente = calcular_semente_execucao(
        config_execucao["execucao"]["seed"],
        indice_instancia,
        indice_execucao,
    )
    gerador_aleatorio = np.random.default_rng(semente)

    motor = PBILFuzzy(
        instancia=instancia,
        config=config_execucao,
        gerador_aleatorio=gerador_aleatorio,
    )

    tempo_inicio = time.time()
    resultado = motor.executar()
    tempo_segundos = time.time() - tempo_inicio

    return {
        "instancia": instancia.nome,
        "numero_jobs": instancia.numero_jobs,
        "numero_maquinas": instancia.numero_maquinas,
        "indice_execucao": indice_execucao,
        "semente": semente,
        "cmax_best": float(resultado.cmax_best),
        "geracoes": len(resultado.historico_cmax_best),
        "tempo_segundos": round(float(tempo_segundos), 3),
        "alpha_medio": float(np.mean(resultado.historico_alpha)),
        "beta_medio": float(np.mean(resultado.historico_beta)),
        "diversidade_media": float(np.mean(resultado.historico_diversidade_estrutural)),
        "historico_cmax_best": resultado.historico_cmax_best,
        "historico_alpha": resultado.historico_alpha,
        "historico_beta": resultado.historico_beta,
        "historico_diversidade_estrutural": resultado.historico_diversidade_estrutural,
    }

In [ ]:
caminho_config = os.path.join(caminho_raiz, "config.yaml")
config = carregar_configuracao(caminho_config)

diretorio_large = os.path.join(caminho_raiz, config["instancia"]["diretorio_large"])
caminhos_instancias_large = listar_instancias(diretorio_large)

print(f"Instâncias Large encontradas: {len(caminhos_instancias_large)}")
print("Primeiras instâncias:")
for caminho in caminhos_instancias_large[:10]:
    print(" -", os.path.basename(caminho))

In [ ]:
max_instancias_large = 10
n_execucoes_por_instancia = 3
max_geracoes_large = 100
n_pop_large = 40

config_large = preparar_config_large(
    config,
    max_geracoes=max_geracoes_large,
    n_pop=n_pop_large,
)

caminhos_selecionados_large = caminhos_instancias_large[:max_instancias_large]

print(f"Instâncias selecionadas: {len(caminhos_selecionados_large)}")
print(f"Execuções por instância: {n_execucoes_por_instancia}")
print(f"Gerações por execução: {config_large['criterio_parada']['max_geracoes']}")
print(f"Tamanho da população: {config_large['pbil']['n_pop']}")

In [ ]:
linhas_resultados_large = []

for indice_instancia, caminho_instancia in enumerate(caminhos_selecionados_large):
    nome_instancia = os.path.splitext(os.path.basename(caminho_instancia))[0]

    for indice_execucao in range(n_execucoes_por_instancia):
        linha_resultado = executar_rodada_large(
            caminho_instancia=caminho_instancia,
            config_execucao=config_large,
            indice_instancia=indice_instancia,
            indice_execucao=indice_execucao,
        )

        linhas_resultados_large.append(linha_resultado)

        print(
            f"{nome_instancia} | execução {indice_execucao + 1}/{n_execucoes_por_instancia} "
            f"| Cmax_best={linha_resultado['cmax_best']:.2f} "
            f"| tempo={linha_resultado['tempo_segundos']:.2f}s"
        )

tabela_resultados_large = pd.DataFrame(linhas_resultados_large)
tabela_resultados_large.head()

In [ ]:
colunas_resumo = [
    "instancia",
    "numero_jobs",
    "numero_maquinas",
    "indice_execucao",
    "semente",
    "cmax_best",
    "geracoes",
    "tempo_segundos",
    "alpha_medio",
    "beta_medio",
    "diversidade_media",
]

tabela_resumo_large = tabela_resultados_large[colunas_resumo].copy()
tabela_resumo_large

In [ ]:
tabela_agregada_large = (
    tabela_resumo_large
    .groupby(["instancia", "numero_jobs", "numero_maquinas"], as_index=False)
    .agg(
        cmax_medio=("cmax_best", "mean"),
        cmax_desvio_padrao=("cmax_best", "std"),
        cmax_melhor=("cmax_best", "min"),
        cmax_pior=("cmax_best", "max"),
        tempo_medio_segundos=("tempo_segundos", "mean"),
        alpha_medio=("alpha_medio", "mean"),
        beta_medio=("beta_medio", "mean"),
        diversidade_media=("diversidade_media", "mean"),
    )
    .sort_values(["numero_jobs", "numero_maquinas", "instancia"])
)

tabela_agregada_large

In [ ]:
melhor_linha = tabela_resultados_large.loc[tabela_resultados_large["cmax_best"].idxmin()]
print("Melhor execução Large")
print(f"Instância: {melhor_linha['instancia']}")
print(f"Execução: {melhor_linha['indice_execucao']}")
print(f"Cmax_best: {melhor_linha['cmax_best']:.2f}")
print(f"Tempo: {melhor_linha['tempo_segundos']:.2f}s")

In [ ]:
visualization.plotar_convergencia_cmax(
    melhor_linha["historico_cmax_best"],
    titulo=f"Convergência do Cmax — {melhor_linha['instancia']}",
)

visualization.plotar_evolucao_alpha_beta(
    melhor_linha["historico_alpha"],
    melhor_linha["historico_beta"],
    titulo=f"Evolução de alpha e beta — {melhor_linha['instancia']}",
)

visualization.plotar_diversidade_estrutural(
    melhor_linha["historico_diversidade_estrutural"],
    titulo=f"Diversidade estrutural — {melhor_linha['instancia']}",
)

In [ ]:
resultados_por_instancia = {
    nome_instancia: grupo["cmax_best"].tolist()
    for nome_instancia, grupo in tabela_resumo_large.groupby("instancia")
}

visualization.plotar_comparacao_boxplot(
    resultados_por_instancia,
    titulo="Comparação do Cmax final — instâncias Large",
)

In [ ]:
diretorio_outputs = os.path.join(caminho_raiz, config["caminhos"]["outputs"])
os.makedirs(diretorio_outputs, exist_ok=True)

caminho_csv_large = os.path.join(diretorio_outputs, "resumo_large.csv")
caminho_json_large = os.path.join(diretorio_outputs, "resumo_large_com_historicos.json")

tabela_resumo_large.to_csv(caminho_csv_large, index=False, encoding="utf-8")

with open(caminho_json_large, "w", encoding="utf-8") as arquivo:
    json.dump(linhas_resultados_large, arquivo, indent=2, ensure_ascii=False)

print(f"Resumo salvo em: {caminho_csv_large}")
print(f"Históricos salvos em: {caminho_json_large}")